In [1]:
import scanpy as sc
import pickle
import os
from scipy.sparse import csr_matrix


In [2]:
rna = sc.read_h5ad("../../data/sea_ad/rna_subset.h5ad")

# 2) Pick the symbol column (feature_names vs feature_name)
name_col = 'feature_names' if 'feature_names' in rna.var.columns else 'feature_name'

# 3) Build a clean SYMBOL column; fall back to Ensembl if missing
sym = rna.var[name_col].astype('string').str.strip()
sym = sym.mask(sym.isna() | (sym == ''), rna.var_names)   # fallback to existing ID if empty
rna.var['symbol'] = sym.str.upper()                       # optional: uppercase for consistency

# 4) Set symbols as var_names and make unique
rna.var_names = rna.var['symbol'].to_numpy()
rna.var_names_make_unique()

# 5) Update raw var_names to match (raw.var_names is read-only, so recreate raw)
if rna.raw is not None:
    # Store the raw layer before overwriting
    raw_adata = rna.raw.to_adata()
    raw_adata.var_names = rna.var_names
    # Clear raw first, then reassign from the fixed adata
    rna.raw = None
    rna.raw = raw_adata

use_hvg = False
if use_hvg:
    sc.pp.highly_variable_genes(rna, n_top_genes=15000, batch_key="donor_id", flavor="seurat")
    hv = rna.var['highly_variable'].astype(bool).to_numpy()   # per-gene boolean
    rna = rna[:, hv].copy()


rna.write_h5ad("data_inputs/SEAAD_MTG_microglia_rna.h5ad")

In [7]:
rna.X.todense()

matrix([[0.       , 0.       , 0.       , ..., 0.       , 0.       ,
         0.       ],
        [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
         0.       ],
        [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
         0.       ],
        ...,
        [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
         0.       ],
        [0.       , 0.       , 1.0314229, ..., 0.       , 0.       ,
         0.       ],
        [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
         0.       ]], dtype=float32)

In [3]:
(rna.var_names == "NFATC2").any()

True

In [6]:
#!/usr/bin/env python

import os
import pickle

import numpy as np
import pandas as pd
import scanpy as sc

from pycisTopic.cistopic_class import create_cistopic_object
from pycisTopic.lda_models import run_cgs_models, evaluate_models
from pycisTopic.topic_binarization import binarize_topics
from pycisTopic.utils import region_names_to_coordinates

# -----------------------
# 0. Paths / basic setup
# -----------------------
# Adjust this to your microglia-only ATAC AnnData:
atac_h5ad = "../../data/sea_ad/atac_subset.h5ad"

out_dir = "outs"
os.makedirs(out_dir, exist_ok=True)

# For SCENIC+ region sets
region_sets_root = os.path.join(out_dir, "region_sets")
topics_top3k_dir = os.path.join(region_sets_root, "Topics_top_3k")
topics_otsu_dir = os.path.join(region_sets_root, "Topics_otsu")
os.makedirs(topics_top3k_dir, exist_ok=True)
os.makedirs(topics_otsu_dir, exist_ok=True)

# -----------------------
# 1. Load microglia ATAC
# -----------------------
print("Loading microglia ATAC AnnData...")
adata_atac = sc.read_h5ad(atac_h5ad)

# At this point we assume:
# - adata_atac is already *microglia-only*
# - adata_atac.var_names are in "chr:start-end" format
print(adata_atac)

# --------------------------------------------
# 2. Create cisTopic object from peak matrix
#    (template-style: DataFrame -> create_cistopic_object)
# --------------------------------------------
print("Building count matrix DataFrame (cells x peaks)...")
# .X might be sparse; toarray() is fine if microglia subset is not huge.
count_array = adata_atac.X.toarray()
count_df = pd.DataFrame(
    data=count_array,
    index=adata_atac.obs_names,   # cells
    columns=adata_atac.var_names  # regions (peaks)
)

# pycisTopic expects regions x cells
count_df = count_df.transpose()   # (regions x cells)

print("Creating CistopicObject...")
cistopic_obj = create_cistopic_object(
    count_df
)
print(cistopic_obj)

# Add cell metadata (e.g., sample, clusters, etc.) from AnnData
print("Adding cell metadata from AnnData.obs...")
cistopic_obj.add_cell_data(adata_atac.obs)

# Save raw cisTopic object (no model yet)
with open(os.path.join(out_dir, "cistopic_obj_raw.pkl"), "wb") as f:
    pickle.dump(cistopic_obj, f)

# -----------------------
# 3. Run LDA models (Python-only CGS)
# -----------------------
print("Running serial LDA models with run_cgs_models (no MALLET)...")

# Reasonable topic range for a single cell type; you can adjust
N_TOPICS = [20]

models = run_cgs_models(
    cistopic_obj=cistopic_obj,
    n_topics=N_TOPICS,
    n_cpu=6,           # limit cores to reduce RAM use
    n_iter=500,        # like the tutorial
    random_state=555,
    alpha=50,
    alpha_by_topic=True,
    eta=0.1,
    eta_by_topic=False,
    save_path=None     # no per-model pickle needed
)

# Save models if you want to inspect later
with open(os.path.join(out_dir, "models.pkl"), "wb") as f:
    pickle.dump(models, f)



/opt/anaconda3/envs/scenicplus/lib/python3.11/site-packages/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/opt/anaconda3/envs/scenicplus/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-08 20:46:04,933	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Loading microglia ATAC AnnData...
AnnData object with n_obs × n_vars = 3129 × 218882
    obs: 'sample_id', 'Neurotypical reference', 'Donor ID', 'Organism', 'Brain Region', 'Sex', 'Gender', 'Age at Death', 'Race (choice=White)', 'Race (choice=Black/ African American)', 'Race (choice=Asian)', 'Race (choice=American Indian/ Alaska Native)', 'Race (choice=Native Hawaiian or Pacific Islander)', 'Race (choice=Unknown or unreported)', 'Race (choice=Other)', 'specify other race', 'Hispanic/Latino', 'Highest level of education', 'Years of education', 'PMI', 'Fresh Brain Weight', 'Brain pH', 'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score', 'Overall CAA Score', 'Highest Lewy Body Disease', 'Total Microinfarcts (not observed grossly)', 'Total microinfarcts in screening sections', 'Atherosclerosis', 'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Last CASI Score', 'Interval from last CASI in months', 'Last MMSE Score', 'Interval from last MMSE in months', 'Last MOCA Score'

2025-12-08 20:46:11,431	INFO worker.py:1724 -- Started a local Ray instance.
(pid=83924) /opt/anaconda3/envs/scenicplus/lib/python3.11/site-packages/pycisTopic/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
(pid=83924)   from pkg_resources import DistributionNotFound, get_distribution


(run_cgs_model pid=83924) 2025-12-08 20:46:13,914 cisTopic     INFO     Running model with 20 topics
(run_cgs_model pid=83924) 2025-12-08 21:10:45,670 cisTopic     INFO     Model with 20 topics done!


In [13]:
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
plt.ioff()  # Turn off interactive mode

# -----------------------
# 4. Model selection
# -----------------------
print("Selecting best model with evaluate_models...")

# If select_model=None, best model is chosen automatically
# based on quality metrics. 
best_model = models[0]
cistopic_obj.add_LDA_model(best_model)

with open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb") as f:
    pickle.dump(cistopic_obj, f)

# -----------------------
# 5. Topic binarization
#    (ntop=3000 and otsu, like the tutorial)
# -----------------------
print("Binarizing topic–region distributions...")

# Top 3k peaks per topic
region_bin_topics_top_3k = binarize_topics(
    cistopic_obj,
    method="ntop",
    ntop=3_000,
    plot=False
)

# Otsu thresholding
region_bin_topics_otsu = binarize_topics(
    cistopic_obj,
    method="otsu",
    plot=False
)

# -----------------------
# 6. Export region sets as BED
#    (SCENIC+ will use these)
# -----------------------
print(f"Exporting region sets to:\n  {topics_top3k_dir}\n  {topics_otsu_dir}")

# Helper to write topic -> BED
def write_topic_beds(region_bin_topics, out_dir):
    for topic_name, regions in region_bin_topics.items():
        # regions.index are strings like "chr:start-end"
        bed_df = region_names_to_coordinates(regions.index)
        # Optional: sort BED
        bed_df = bed_df.sort_values(["Chromosome", "Start", "End"])
        bed_path = os.path.join(out_dir, f"{topic_name}.bed")
        bed_df.to_csv(bed_path, sep="\t", header=False, index=False)

# Write ntop=3k sets
write_topic_beds(region_bin_topics_top_3k, topics_top3k_dir)

# Write otsu sets
write_topic_beds(region_bin_topics_otsu, topics_otsu_dir)

print("All done 🎉")
print(f"- cisTopic object with model: {os.path.join(out_dir, 'cistopic_obj.pkl')}")
print(f"- Region sets (ntop):        {topics_top3k_dir}")
print(f"- Region sets (otsu):        {topics_otsu_dir}")
print("Use these paths in your SCENIC+ Snakemake config (region_set_folder = 'outs/region_sets').")


Selecting best model with evaluate_models...
Binarizing topic–region distributions...
Exporting region sets to:
  outs/region_sets/Topics_top_3k
  outs/region_sets/Topics_otsu
All done 🎉
- cisTopic object with model: outs/cistopic_obj.pkl
- Region sets (ntop):        outs/region_sets/Topics_top_3k
- Region sets (otsu):        outs/region_sets/Topics_otsu
Use these paths in your SCENIC+ Snakemake config (region_set_folder = 'outs/region_sets').


In [ ]:
!mkdir -p scplus_pipeline
!scenicplus init_snakemake --out_dir scplus_pipeline

In [ ]:
# Copy cisTopic outputs to data_inputs so Snakemake config can find them
import shutil
data_inputs = "data_inputs"
os.makedirs(data_inputs, exist_ok=True)
shutil.copy(os.path.join(out_dir, "cistopic_obj.pkl"), os.path.join(data_inputs, "cistopic_obj.pkl"))
if os.path.exists(os.path.join(data_inputs, "region_sets")):
    shutil.rmtree(os.path.join(data_inputs, "region_sets"))
shutil.copytree(os.path.join(out_dir, "region_sets"), os.path.join(data_inputs, "region_sets"))
print("Copied cistopic_obj.pkl and region_sets to data_inputs/")

In [ ]:

!mkdir -p tmp

In [ ]:
import pandas as pd

# Use chromosome names WITH 'chr' prefix to match ATAC data (chr4:16412872-16415576)
chrom_sizes = {
    "chr1": 248956422,
    "chr2": 242193529,
    "chr3": 198295559,
    "chr4": 190214555,
    "chr5": 181538259,
    "chr6": 170805979,
    "chr7": 159345973,
    "chr8": 145138636,
    "chr9": 138394717,
    "chr10": 133797422,
    "chr11": 135086622,
    "chr12": 133275309,
    "chr13": 114364328,
    "chr14": 107043718,
    "chr15": 101991189,
    "chr16": 90338345,
    "chr17": 83257441,
    "chr18": 80373285,
    "chr19": 58617616,
    "chr20": 64444167,
    "chr21": 46709983,
    "chr22": 50818468,
    "chrX": 156040895,
    "chrY": 57227415,
    "chrM": 16569,
}

df = pd.DataFrame({
    "Chromosome": list(chrom_sizes.keys()),
    "Start": [0] * len(chrom_sizes),
    "End": list(chrom_sizes.values()),
})

out_path = "../../code/05_scenicplus/scplus_pipeline/Snakemake/chromsizes.tsv"
df.to_csv(out_path, sep="\t", index=False)
print("Wrote:", out_path)

In [ ]:
import pandas as pd

# Fix genome_annotation.tsv - add 'chr' prefix to match ATAC data
ga_path = "../../code/05_scenicplus/scplus_pipeline/Snakemake/genome_annotation.tsv"
ga = pd.read_table(ga_path, dtype={'Chromosome': str})

# Map chromosome names to chr-prefixed versions
def add_chr_prefix(chrom):
    chrom = str(chrom)
    # Skip if already has chr prefix
    if chrom.startswith('chr'):
        return chrom
    # Map MT to chrM
    if chrom == 'MT':
        return 'chrM'
    # Standard chromosomes (1-22, X, Y)
    if chrom in [str(i) for i in range(1, 23)] + ['X', 'Y']:
        return f'chr{chrom}'
    # Other contigs - skip them (they won't match ATAC anyway)
    return chrom

ga['Chromosome'] = ga['Chromosome'].apply(add_chr_prefix)

# Save fixed file
ga.to_csv(ga_path, sep="\t", index=False)
print(f"Fixed genome_annotation.tsv - {len(ga)} rows")
print(f"Unique chromosomes (sample): {ga['Chromosome'].unique()[:10]}")

In [ ]:
project_dir = "../../code/05_scenicplus/scplus_pipeline/Snakemake"

%cd "{project_dir}"

!snakemake --cores 6 --rerun-incomplete

../../code/05_scenicplus/scplus_pipeline/Snakemake
Assuming unrestricted shared filesystem usage for local execution.
Building DAG of jobs...
Assuming unrestricted shared filesystem usage for local execution.
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 6
Rules claiming more threads will be scaled down.
Job stats:
job                           count
--------------------------  -------
AUCell_direct                     1
AUCell_extended                   1
all                               1
eGRN_direct                       1
eGRN_extended                     1
get_search_space                  1
motif_enrichment_cistarget        1
motif_enrichment_dem              1
prepare_GEX_ACC_multiome          1
prepare_menr                      1
region_to_gene                    1
scplus_mudata                     1
tf_to_gene                        1
total                            13

Select jobs to execute...
Execute 1 jobs...

[Mon Dec  8 22:15:15 2025]
localrule motif_e

In [ ]:
import mudata as md
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import linregress
from statsmodels.stats.multitest import multipletests

# ========= 1. Load data =========
mdata = md.read_h5mu("scplus_pipeline/Snakemake/scplusmdata.h5mu")

# regulon activity: extended gene-based AUC
ra = mdata.mod["extended_gene_based_AUC"]  # cells × regulons
X = ra.X.toarray() if issparse(ra.X) else ra.X
reg_names = np.array(ra.var_names)

# RNA metadata (for ADNC + donor_id)
rna_obs = mdata.mod["scRNA_counts"].obs

# sanity check (optional)
print("n_cells RA:", X.shape[0], "n_cells RNA:", rna_obs.shape[0])
assert np.all(ra.obs_names == rna_obs.index), "Cell order mismatch!"

# ========= 2. Build cell-level DataFrame =========
df = pd.DataFrame(X, index=ra.obs_names, columns=reg_names)
df["donor_id"] = rna_obs["donor_id"].values
df["ADNC"] = rna_obs["ADNC"].values

# ========= 3. Pseudobulk per donor =========
# mean regulon AUCell per donor
bulk_ra = df.groupby("donor_id")[reg_names].mean()   # donors × regulons

# ADNC per donor (take first value for each donor)
adnc_by_donor = df.groupby("donor_id")["ADNC"].first()

# align ADNC with bulk_ra rows
adnc_by_donor = adnc_by_donor.loc[bulk_ra.index]

# encode ADNC as ordered categorical
print("ADNC categories:", adnc_by_donor.unique())

order = ["Not AD", "Low", "Intermediate", "High"]  # change if your labels differ
adnc_cat = pd.Categorical(adnc_by_donor, categories=order, ordered=True)
stage = adnc_cat.codes.astype(float)   # -1 means "not in categories"

# keep donors with valid ADNC
mask = stage >= 0
bulk_ra = bulk_ra[mask]
stage = stage[mask]

print("n_donors used:", bulk_ra.shape[0])

X_bulk = bulk_ra.values   # donors × regulons

# ========= 4. Regression AUCell ~ ADNC (donor-level) =========
results = []
for i, name in enumerate(reg_names):
    y = X_bulk[:, i]
    slope, intercept, r, p, stderr = linregress(stage, y)
    results.append((name, slope, r, p))

res = pd.DataFrame(results, columns=["regulon", "slope", "r", "p"])

# FDR correction
res["padj"] = multipletests(res["p"], method="fdr_bh")[1]
res["abs_slope"] = res["slope"].abs()

# ========= 5. Inspect / save results =========
# top ADNC-associated regulons (by FDR then effect s

res[res["padj"] <= 0.05]


In [ ]:
import scanpy as sc

gene_auc = md.read("scplus_pipeline/Snakemake/AUCell_direct.h5mu").mod["Gene_based"]
atac_auc = md.read("scplus_pipeline/Snakemake/AUCell_direct.h5mu").mod["Region_based"]

In [ ]:
# Calculate mean AUCell scores across all cells for each TF (gene-based regulons)
mean_auc_scores = gene_auc.X.mean(axis=0)

# Handle sparse matrix if needed
import numpy as np
if hasattr(mean_auc_scores, 'A1'):
    mean_auc_scores = mean_auc_scores.A1

# Create a DataFrame with TF names and their mean AUCell scores
tf_scores = pd.DataFrame({
    'TF': gene_auc.var_names,
    'mean_AUCell': mean_auc_scores
}).sort_values('mean_AUCell', ascending=False)

# Display top 20 TFs with highest AUCell scores
print("Top 20 TFs with highest mean AUCell scores:")
tf_scores

In [ ]:
temp = sc.read("data_inputs/SEAAD_MTG_microglia_rna.h5ad")
temp.raw.X.todense().astype(int)

In [ ]:
input_data = sc.read("../../code/05_scenicplus/data_inputs/SEAAD_MTG_microglia_rna.h5ad")
input_data.var.loc["BHLHE41"]